# AIC 2026 — Merge 4 Parts (Simple & Stable)

Notebook này chỉ làm 4 việc:

1. Đọc output của Part 1–4 từ đúng path đã cung cấp.
2. Kiểm tra schema và dữ liệu trùng.
3. Gộp manifest.
4. Xuất một bộ Batch 1 thống nhất.

Không tự dò path, không suy đoán tên biến, không giải nén ZIP.

In [1]:
# Cell 1 — Import
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
# Cell 2 — Đường dẫn cố định
PART_ROOTS = {
    "part1": Path("/kaggle/input/notebooks/jamel27/aic2026-batch1-part1-preprocession"),
    "part2": Path("/kaggle/input/notebooks/jamel27/aic2026-batch1-part2-preprocession"),
    "part3": Path("/kaggle/input/notebooks/minhdat27/aic2026-batch1-part3-preprocession"),
    "part4": Path("/kaggle/input/notebooks/nddttt/aic2026-batch1-part4-preprocess"),
}

OUTPUT_DIR = Path("/kaggle/working/aic2026_batch1_merged")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for part, root in PART_ROOTS.items():
    if not root.exists():
        raise FileNotFoundError(f"Không tìm thấy {part}: {root}")
    print(part, "->", root)


part1 -> /kaggle/input/notebooks/jamel27/aic2026-batch1-part1-preprocession
part2 -> /kaggle/input/notebooks/jamel27/aic2026-batch1-part2-preprocession
part3 -> /kaggle/input/notebooks/minhdat27/aic2026-batch1-part3-preprocession
part4 -> /kaggle/input/notebooks/nddttt/aic2026-batch1-part4-preprocess


In [3]:
# Cell 3 — Tìm đúng file trong từng output
def find_required_file(root: Path, filename: str) -> Path:
    matches = sorted(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Không tìm thấy {filename} trong {root}")
    if len(matches) > 1:
        print(f"Cảnh báo: tìm thấy nhiều {filename} trong {root}, dùng file đầu tiên:")
        for p in matches:
            print(" -", p)
    return matches[0]

files = {}

for part, root in PART_ROOTS.items():
    files[part] = {
        "manifest_keyframes": find_required_file(root, "manifest_keyframes.parquet"),
        "manifest_videos": find_required_file(root, "manifest_videos.parquet"),
        "feature_catalog": find_required_file(root, "feature_catalog.parquet"),
    }

    error_matches = sorted(root.rglob("errors.csv"))
    files[part]["errors"] = error_matches[0] if error_matches else None

display(pd.DataFrame([
    {
        "part": part,
        "manifest_keyframes": str(paths["manifest_keyframes"]),
        "manifest_videos": str(paths["manifest_videos"]),
        "feature_catalog": str(paths["feature_catalog"]),
        "errors": str(paths["errors"]) if paths["errors"] else None,
    }
    for part, paths in files.items()
]))


,part,manifest_keyframes,manifest_videos,feature_catalog,errors
0,part1,/kaggle/input/notebooks/jamel27/aic2026-batch1...,/kaggle/input/notebooks/jamel27/aic2026-batch1...,/kaggle/input/notebooks/jamel27/aic2026-batch1...,/kaggle/input/notebooks/jamel27/aic2026-batch1...
1,part2,/kaggle/input/notebooks/jamel27/aic2026-batch1...,/kaggle/input/notebooks/jamel27/aic2026-batch1...,/kaggle/input/notebooks/jamel27/aic2026-batch1...,/kaggle/input/notebooks/jamel27/aic2026-batch1...
2,part3,/kaggle/input/notebooks/minhdat27/aic2026-batc...,/kaggle/input/notebooks/minhdat27/aic2026-batc...,/kaggle/input/notebooks/minhdat27/aic2026-batc...,/kaggle/input/notebooks/minhdat27/aic2026-batc...
3,part4,/kaggle/input/notebooks/nddttt/aic2026-batch1-...,/kaggle/input/notebooks/nddttt/aic2026-batch1-...,/kaggle/input/notebooks/nddttt/aic2026-batch1-...,/kaggle/input/notebooks/nddttt/aic2026-batch1-...


In [4]:
# Cell 4 — Đọc dữ liệu
manifests = {}
videos = {}
features = {}
errors_by_part = {}

for part, paths in files.items():
    manifests[part] = pd.read_parquet(paths["manifest_keyframes"])
    videos[part] = pd.read_parquet(paths["manifest_videos"])
    features[part] = pd.read_parquet(paths["feature_catalog"])

    if paths["errors"] is not None:
        errors_by_part[part] = pd.read_csv(paths["errors"])
    else:
        errors_by_part[part] = pd.DataFrame()

    # Ép source_part đúng theo input hiện tại.
    manifests[part]["source_part"] = part
    videos[part]["source_part"] = part
    features[part]["source_part"] = part

    print(
        part,
        "| keyframes:", len(manifests[part]),
        "| videos:", len(videos[part]),
        "| features:", len(features[part]),
    )


part1 | keyframes: 53943 | videos: 187 | features: 187
part2 | keyframes: 36378 | videos: 191 | features: 191
part3 | keyframes: 40319 | videos: 171 | features: 171
part4 | keyframes: 46681 | videos: 324 | features: 324


In [5]:
# Cell 5 — Kiểm tra schema
reference_part = "part1"

def compare_schema(reference: pd.DataFrame, current: pd.DataFrame):
    ref_cols = list(reference.columns)
    cur_cols = list(current.columns)

    missing = sorted(set(ref_cols) - set(cur_cols))
    extra = sorted(set(cur_cols) - set(ref_cols))
    return missing, extra

schema_problems = []

for part in ["part2", "part3", "part4"]:
    missing, extra = compare_schema(manifests[reference_part], manifests[part])
    if missing or extra:
        schema_problems.append({
            "part": part,
            "missing_columns": missing,
            "extra_columns": extra,
        })

if schema_problems:
    display(pd.DataFrame(schema_problems))
    raise RuntimeError("Schema manifest giữa 4 Part chưa giống nhau.")

print("Schema manifest của 4 Part giống nhau.")


Schema manifest của 4 Part giống nhau.


In [6]:
# Cell 6 — Gộp dữ liệu
manifest_all = pd.concat(
    [manifests[p] for p in ["part1", "part2", "part3", "part4"]],
    ignore_index=True,
    sort=False,
)

videos_all = pd.concat(
    [videos[p] for p in ["part1", "part2", "part3", "part4"]],
    ignore_index=True,
    sort=False,
)

features_all = pd.concat(
    [features[p] for p in ["part1", "part2", "part3", "part4"]],
    ignore_index=True,
    sort=False,
)

error_frames = []
for part, df in errors_by_part.items():
    if not df.empty:
        temp = df.copy()
        temp["source_part"] = part
        error_frames.append(temp)

errors_all = (
    pd.concat(error_frames, ignore_index=True, sort=False)
    if error_frames
    else pd.DataFrame()
)

# Xóa global_id cũ vì mỗi Part đều bắt đầu từ 0.
if "global_id" in manifest_all.columns:
    manifest_all = manifest_all.drop(columns=["global_id"])

# Tạo stable key nếu notebook preprocess chưa có.
if "global_key" not in manifest_all.columns:
    manifest_all["global_key"] = (
        manifest_all["source_part"].astype(str)
        + ":"
        + manifest_all["video_id"].astype(str)
        + ":"
        + manifest_all["feature_row"].astype(str)
    )

manifest_all = (
    manifest_all
    .sort_values(["source_part", "video_id", "feature_row"])
    .reset_index(drop=True)
)

manifest_all.insert(
    0,
    "global_id",
    np.arange(len(manifest_all), dtype=np.int64),
)

videos_all = (
    videos_all
    .sort_values(["source_part", "video_id"])
    .reset_index(drop=True)
)

features_all = (
    features_all
    .sort_values(["source_part", "video_id"])
    .reset_index(drop=True)
)

print("Merged keyframes:", f"{len(manifest_all):,}")
print("Merged videos:", f"{len(videos_all):,}")
print("Merged feature files:", f"{len(features_all):,}")


Merged keyframes: 177,321
Merged videos: 873
Merged feature files: 873


In [7]:
# Cell 7 — Kiểm tra sau merge
checks = {
    "global_id_unique": bool(manifest_all["global_id"].is_unique),
    "global_key_unique": bool(manifest_all["global_key"].is_unique),
    "vector_locator_unique": bool(
        ~manifest_all.duplicated(
            ["source_part", "video_id", "feature_row"]
        ).any()
    ),
    "all_4_parts_present": set(manifest_all["source_part"].unique())
        == {"part1", "part2", "part3", "part4"},
}

if "embedding_dim" in manifest_all.columns:
    checks["embedding_dim_is_512"] = bool(
        manifest_all["embedding_dim"].dropna().eq(512).all()
    )

if "alignment_ok" in videos_all.columns:
    checks["all_video_alignment_ok"] = bool(
        videos_all["alignment_ok"].fillna(False).all()
    )

# Kiểm tra video_id có xuất hiện ở nhiều Part không.
video_part_count = (
    videos_all.groupby("video_id")["source_part"].nunique()
)
duplicate_video_ids = video_part_count[video_part_count > 1]
checks["video_id_not_shared_across_parts"] = bool(duplicate_video_ids.empty)

print(json.dumps(checks, ensure_ascii=False, indent=2))

if not checks["global_key_unique"]:
    duplicates = manifest_all[
        manifest_all.duplicated("global_key", keep=False)
    ][["global_key", "source_part", "video_id", "feature_row"]]
    display(duplicates.head(50))
    raise RuntimeError("global_key bị trùng.")

if not checks["vector_locator_unique"]:
    raise RuntimeError(
        "Trùng source_part + video_id + feature_row."
    )

if not duplicate_video_ids.empty:
    display(duplicate_video_ids.head(50))
    raise RuntimeError(
        "Có video_id xuất hiện ở nhiều Part."
    )


{
  "global_id_unique": true,
  "global_key_unique": true,
  "vector_locator_unique": true,
  "all_4_parts_present": true,
  "embedding_dim_is_512": true,
  "all_video_alignment_ok": true,
  "video_id_not_shared_across_parts": true
}


In [8]:
# Cell 8 — Xuất output
manifest_path = OUTPUT_DIR / "manifest_keyframes.parquet"
videos_path = OUTPUT_DIR / "manifest_videos.parquet"
features_path = OUTPUT_DIR / "feature_catalog.parquet"
errors_path = OUTPUT_DIR / "errors.csv"
sample_path = OUTPUT_DIR / "manifest_sample.csv"
summary_path = OUTPUT_DIR / "summary.json"
schema_path = OUTPUT_DIR / "schema.json"

manifest_all.to_parquet(
    manifest_path,
    index=False,
    compression="zstd",
)
videos_all.to_parquet(
    videos_path,
    index=False,
    compression="zstd",
)
features_all.to_parquet(
    features_path,
    index=False,
    compression="zstd",
)

if errors_all.empty:
    pd.DataFrame(
        columns=[
            "severity", "stage", "video_id",
            "asset", "error_type", "detail", "source_part",
        ]
    ).to_csv(errors_path, index=False, encoding="utf-8-sig")
else:
    errors_all.to_csv(
        errors_path,
        index=False,
        encoding="utf-8-sig",
    )

manifest_all.head(500).to_csv(
    sample_path,
    index=False,
    encoding="utf-8-sig",
)

summary = {
    "schema_version": "aic2026-batch1-merged-v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_keyframes": int(len(manifest_all)),
    "total_videos": int(len(videos_all)),
    "total_feature_files": int(len(features_all)),
    "keyframes_by_part": {
        str(k): int(v)
        for k, v in manifest_all.groupby("source_part").size().items()
    },
    "videos_by_part": {
        str(k): int(v)
        for k, v in videos_all.groupby("source_part").size().items()
    },
    "checks": checks,
}

schema = {
    "schema_version": "aic2026-batch1-merged-v1",
    "main_file": "manifest_keyframes.parquet",
    "primary_key": "global_id",
    "stable_key": "global_key",
    "vector_locator": [
        "source_part",
        "clip_feature_relpath",
        "feature_row",
    ],
    "submission_mapping": [
        "video_id",
        "frame_id",
    ],
    "preview_mapping": [
        "source_part",
        "keyframe_relpath",
        "video_relpath",
        "timestamp_sec",
    ],
    "manifest_keyframes_columns": {
        col: str(dtype)
        for col, dtype in manifest_all.dtypes.items()
    },
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with schema_path.open("w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print("Output:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

print("\nMain output:", manifest_path)


Output:
 - errors.csv
 - feature_catalog.parquet
 - manifest_keyframes.parquet
 - manifest_sample.csv
 - manifest_videos.parquet
 - schema.json
 - summary.json

Main output: /kaggle/working/aic2026_batch1_merged/manifest_keyframes.parquet


## Kết quả

Sau khi chạy xong, tạo Kaggle Dataset từ:

```text
/kaggle/working/aic2026_batch1_merged/
```

Dataset này là Data Layer thống nhất cho bước build FAISS/Milvus tiếp theo.